In [25]:
import pandas as pd
import numpy as np
import itertools

In [14]:
import json
import random
from collections import Counter
from itertools import combinations
from pathlib import Path

sequences = ['13524232514', '54231251343', '21435235214', '34352452141', '42531451342', '31525423241']

# Positions 0,2 = speech-trained; 1,3 = finger-trained; 4,5 = untrained.
# Subjects 1-5 already collected — keep their original group assignments fixed.
locked_perms = [
    ('13524232514', '54231251343', '21435235214', '34352452141', '42531451342', '31525423241'),
    ('13524232514', '54231251343', '21435235214', '42531451342', '34352452141', '31525423241'),
    ('13524232514', '54231251343', '21435235214', '31525423241', '34352452141', '42531451342'),
    ('13524232514', '54231251343', '34352452141', '21435235214', '42531451342', '31525423241'),
    ('13524232514', '54231251343', '34352452141', '42531451342', '21435235214', '31525423241'),
]

ROLE_BY_POS = {0: 'speech', 2: 'speech', 1: 'finger', 3: 'finger', 4: 'untrained', 5: 'untrained'}
ASSIGNMENT_SEED = 42  # reproducible choice among equally good candidates
n_collected = len(locked_perms)  # do not overwrite TargetFiles for these subjects


def _role_counts(perms):
    counts = Counter()
    for perm in perms:
        for i, seq in enumerate(perm):
            counts[(seq, ROLE_BY_POS[i])] += 1
    return counts


def _assignment_score(counts, speech, finger, untrained):
    """Prefer putting sequences into roles they have under-filled so far."""
    score = 0
    for s in speech:
        score += counts[(s, 'speech')]
    for s in finger:
        score += counts[(s, 'finger')]
    for s in untrained:
        score += counts[(s, 'untrained')]
    trial = counts.copy()
    for s in speech:
        trial[(s, 'speech')] += 1
    for s in finger:
        trial[(s, 'finger')] += 1
    for s in untrained:
        trial[(s, 'untrained')] += 1
    vals = [trial[(s, r)] for s in sequences for r in ('speech', 'finger', 'untrained')]
    mean = sum(vals) / len(vals)
    var = sum((v - mean) ** 2 for v in vals) / len(vals)
    return (score, var)


def build_balanced_perms(n_total, seed=ASSIGNMENT_SEED):
    """Build n_total group assignments: lock collected subjects, then greedily balance roles."""
    rng = random.Random(seed)
    perms = list(locked_perms)
    used_keys = {p[:4] + tuple(sorted(p[4:])) for p in perms}

    while len(perms) < n_total:
        counts = _role_counts(perms)
        candidates = []
        for speech in combinations(sequences, 2):
            rem = [s for s in sequences if s not in speech]
            for finger in combinations(rem, 2):
                untrained = tuple(s for s in rem if s not in finger)
                sc = _assignment_score(counts, speech, finger, untrained)
                for s_ord in itertools.permutations(speech):
                    for f_ord in itertools.permutations(finger):
                        for u_ord in itertools.permutations(untrained):
                            perm = (s_ord[0], f_ord[0], s_ord[1], f_ord[1], u_ord[0], u_ord[1])
                            key = perm[:4] + tuple(sorted(perm[4:]))
                            if key in used_keys:
                                continue
                            candidates.append((sc, perm, key))
        candidates.sort(key=lambda x: x[0])
        best_sc = candidates[0][0]
        best = [c for c in candidates if c[0] == best_sc]
        _, perm, key = rng.choice(best)
        used_keys.add(key)
        perms.append(perm)
    return perms


fingers = ['1', '2', '3', '4', '5']
effectors = [0, 2]  # 0 for speech, 2 for right hand

iti = 2000  # msecs for inter-trial interval
execTime = 5000  # msecs for each trial maximum
precue_time = 2000  # msecs for precue time

total_sub_num = 20

num_familiarization_blocks = 1
num_pre_blocks = 3
num_post_blocks = 3
num_induction_blocks = 6

num_days = 2

effector_symbols = ['#', '$']

unique_perms = build_balanced_perms(total_sub_num)

# Shared with utils.py so analysis labels stay aligned with TargetFiles
assignments_path = Path('group_sequence_assignments.json')
assignments_path.write_text(json.dumps([list(p) for p in unique_perms], indent=2))
print(f"Wrote {assignments_path} ({len(unique_perms)} groups)")

print("\nPer-subject role assignment (speech / finger / untrained):")
for i, perm in enumerate(unique_perms):
    speech = [perm[0], perm[2]]
    finger = [perm[1], perm[3]]
    untrained = [perm[4], perm[5]]
    tag = 'LOCKED' if i < n_collected else 'new'
    print(f"  subj {i+1:2d} [{tag:6s}]: speech={speech}  finger={finger}  untrained={untrained}")

counts = _role_counts(unique_perms)
print("\nRole balance across subjects (cols = times a seq is assigned that role):")
print(f"{'seq':12}  speech  finger  untrained")
for s in sequences:
    print(f"{s}  {counts[(s, 'speech')]:5d}  {counts[(s, 'finger')]:6d}  {counts[(s, 'untrained')]:8d}")


Wrote group_sequence_assignments.json (20 groups)

Per-subject role assignment (speech / finger / untrained):
  subj  1 [LOCKED]: speech=['13524232514', '21435235214']  finger=['54231251343', '34352452141']  untrained=['42531451342', '31525423241']
  subj  2 [LOCKED]: speech=['13524232514', '21435235214']  finger=['54231251343', '42531451342']  untrained=['34352452141', '31525423241']
  subj  3 [LOCKED]: speech=['13524232514', '21435235214']  finger=['54231251343', '31525423241']  untrained=['34352452141', '42531451342']
  subj  4 [LOCKED]: speech=['13524232514', '34352452141']  finger=['54231251343', '21435235214']  untrained=['42531451342', '31525423241']
  subj  5 [LOCKED]: speech=['13524232514', '34352452141']  finger=['54231251343', '42531451342']  untrained=['21435235214', '31525423241']
  subj  6 [new   ]: speech=['42531451342', '31525423241']  finger=['34352452141', '13524232514']  untrained=['21435235214', '54231251343']
  subj  7 [new   ]: speech=['54231251343', '42531451342'

In [20]:
counts = _role_counts(unique_perms[:8])
print("\nRole balance across subjects (cols = times a seq is assigned that role):")
print(f"{'seq':12}  speech  finger  untrained")
for s in sequences:
    print(f"{s}  {counts[(s, 'speech')]:5d}  {counts[(s, 'finger')]:6d}  {counts[(s, 'untrained')]:8d}")


Role balance across subjects (cols = times a seq is assigned that role):
seq           speech  finger  untrained
13524232514      5       1         2
54231251343      1       5         2
21435235214      3       3         2
34352452141      3       2         3
42531451342      2       3         3
31525423241      2       2         4


In [21]:
def check_run(seq):
    #check runs of 3
    for pos in range(len(seq)- 2):
        if (int(seq[pos]) == int(seq[pos + 1]) + 1 and int(seq[pos+1]) == int(seq[pos+2]) + 1):
            return True
        if (int(seq[pos]) == int(seq[pos + 1]) - 1 and int(seq[pos+1]) == int(seq[pos+2]) - 1):
            return True
    
    #check double press
    
    for pos in range(len(seq) - 1):
        if(int(seq[pos]) == int(seq[pos+1])):
            return True


    for finger in fingers:
        if seq.count(finger) == 0:
            return True
        
    # each finger should be used at least twice
    for finger in fingers:
        if seq.count(finger) < 2:
            return True
        
        
    return False

In [7]:
import random
from collections import Counter

seqs = ['13524232514']
print("Generated sequence:", seqs[0])
num_seqs_to_generate = 5

for _ in range(num_seqs_to_generate):

    # Compute zero-order statistics (digit frequencies) for each sequence
    zero_orders = [Counter(seq) for seq in seqs]

    # Compute first-order statistics (transition frequencies) for each sequence
    first_orders = [Counter(zip(seq, seq[1:])) for seq in seqs]

    # Generate a new sequence that matches the statistics
    while True:
        new_seq = ''.join(random.choice('12345') for _ in range(11))
        if check_run(new_seq):
            continue
        new_zero = Counter(new_seq)
        new_first = Counter(zip(new_seq, new_seq[1:]))
        if new_first not in first_orders:
            break
        # if new_zero not in zero_orders and new_first not in first_orders:
        #     break
    
    print("Generated sequence:", new_seq)
    # Add the new sequence to the list of sequences and their statistics
    seqs.append(new_seq)


# generate all zero order and first order stats of the seqs
zero_orders = [Counter(seq)for seq in seqs]
first_orders = [Counter(zip(seq, seq[1:])) for seq in seqs]
# sort them by digit or transition pair
print("Zero-order statistics (digit frequencies):")
for i, zero in enumerate(zero_orders):
    print(f"Sequence {i+1}: {dict(sorted(zero.items()))}")
print("\nFirst-order statistics (transition frequencies):")
for i, first in enumerate(first_orders):
    print(f"Sequence {i+1}: {dict(sorted(first.items()))}")


Generated sequence: 13524232514
Generated sequence: 24352354151
Generated sequence: 52134253415
Generated sequence: 45415324231
Generated sequence: 25325145314
Generated sequence: 43415215232
Zero-order statistics (digit frequencies):
Sequence 1: {'1': 2, '2': 3, '3': 2, '4': 2, '5': 2}
Sequence 2: {'1': 2, '2': 2, '3': 2, '4': 2, '5': 3}
Sequence 3: {'1': 2, '2': 2, '3': 2, '4': 2, '5': 3}
Sequence 4: {'1': 2, '2': 2, '3': 2, '4': 3, '5': 2}
Sequence 5: {'1': 2, '2': 2, '3': 2, '4': 2, '5': 3}
Sequence 6: {'1': 2, '2': 3, '3': 2, '4': 2, '5': 2}

First-order statistics (transition frequencies):
Sequence 1: {('1', '3'): 1, ('1', '4'): 1, ('2', '3'): 1, ('2', '4'): 1, ('2', '5'): 1, ('3', '2'): 1, ('3', '5'): 1, ('4', '2'): 1, ('5', '1'): 1, ('5', '2'): 1}
Sequence 2: {('1', '5'): 1, ('2', '3'): 1, ('2', '4'): 1, ('3', '5'): 2, ('4', '1'): 1, ('4', '3'): 1, ('5', '1'): 1, ('5', '2'): 1, ('5', '4'): 1}
Sequence 3: {('1', '3'): 1, ('1', '5'): 1, ('2', '1'): 1, ('2', '5'): 1, ('3', '4'): 2

In [23]:
### target file headers: [subNum, group, effector, symbol, isTrain, seq, fixedDur, execTime, iti, precueTime]
# effector = 2 for right hand, 0 for speech
# Subjects 1–n_collected already run — leave their TargetFiles untouched.

target_dir = Path('TargetFiles')
target_dir.mkdir(exist_ok=True)
np.random.seed(ASSIGNMENT_SEED)

for sub in range(total_sub_num):
    subNum = sub + 1
    if subNum <= n_collected:
        continue

    group = (subNum - 1) % len(unique_perms)
    block_num = 1

    ################### Familiarization  #######################
    isTrain = 0
    num_rep = 10
    for _ in range(num_familiarization_blocks):
        block = pd.DataFrame(columns=["subNum", "group", "effector", "symbol", "isTrain", "seq", "fixedDur", "execTime", "iti", "precueTime"])
        for effector in effectors:
            for _ in range(num_rep):
                while True:
                    seq = ''.join(map(str, np.random.choice(range(1, 6), size=11)))
                    if not check_run(seq):
                        break
                if effector == 0:
                    symbol = '$'  # symbol for speech
                    fixed_dur = 1  # fixed duration for speech
                else:
                    symbol = '#'  # symbol for right hand
                    fixed_dur = 0
                trial = pd.DataFrame([[subNum, group, effector, symbol, isTrain, seq, fixed_dur, execTime, iti, precue_time]], columns=block.columns)
                block = pd.concat([block, trial], ignore_index=True)

        block = block.sample(frac=1).reset_index(drop=True)
        block.to_csv(target_dir / f'SFS2_subj{subNum}_group{group+1}_block{block_num}.tgt', index=False, sep='\t')
        block_num += 1

    for day in range(num_days):
        ################### preInduction  #######################
        isTrain = 0
        num_rep = 3
        for _ in range(num_pre_blocks):
            block = pd.DataFrame(columns=["subNum", "group", "effector", "symbol", "isTrain", "seq", "fixedDur", "execTime", "iti", "precueTime"])
            for effector in effectors:
                for idx, seq in enumerate(unique_perms[group]):
                    if effector == 0:
                        symbol = '$'
                        fixed_dur = 1
                    else:
                        symbol = '#'
                        fixed_dur = 0
                    trial = pd.DataFrame([[subNum, group, effector, symbol, isTrain, seq, fixed_dur, execTime, iti, precue_time]], columns=block.columns)
                    block = pd.concat([block, trial], ignore_index=True)
            block = pd.concat([block] * num_rep, ignore_index=True)
            block = block.sample(frac=1).reset_index(drop=True)
            block.to_csv(target_dir / f'SFS2_subj{subNum}_group{group+1}_block{block_num}.tgt', index=False, sep='\t')
            block_num += 1

        ####################  Induction  ######################
        isTrain = 1
        num_rep = 8
        for _ in range(num_induction_blocks):
            block = pd.DataFrame(columns=["subNum", "group", "effector", "symbol", "isTrain", "seq", "fixedDur", "execTime", "iti", "precueTime"])
            for idx, seq in enumerate(unique_perms[group][:4]):
                effector = effectors[idx % 2]
                if effector == 0:
                    symbol = '$'
                    fixed_dur = 1
                else:
                    symbol = '#'
                    fixed_dur = 0
                trial = pd.DataFrame([[subNum, group, effector, symbol, isTrain, seq, fixed_dur, execTime, iti, precue_time]], columns=block.columns)
                block = pd.concat([block, trial], ignore_index=True)

            block = pd.concat([block] * num_rep, ignore_index=True)
            block = block.sample(frac=1).reset_index(drop=True)
            block.to_csv(target_dir / f'SFS2_subj{subNum}_group{group+1}_block{block_num}.tgt', index=False, sep='\t')
            block_num += 1

        #################### Post Induction ###########################
        isTrain = 0
        num_rep = 3
        for _ in range(num_post_blocks):
            block = pd.DataFrame(columns=["subNum", "group", "effector", "symbol", "isTrain", "seq", "fixedDur", "execTime", "iti", "precueTime"])
            for effector in effectors:
                for idx, seq in enumerate(unique_perms[group]):
                    if effector == 0:
                        symbol = '$'
                        fixed_dur = 1
                    else:
                        symbol = '#'
                        fixed_dur = 0
                    trial = pd.DataFrame([[subNum, group, effector, symbol, isTrain, seq, fixed_dur, execTime, iti, precue_time]], columns=block.columns)
                    block = pd.concat([block, trial], ignore_index=True)
            block = pd.concat([block] * num_rep, ignore_index=True)
            block = block.sample(frac=1).reset_index(drop=True)
            block.to_csv(target_dir / f'SFS2_subj{subNum}_group{group+1}_block{block_num}.tgt', index=False, sep='\t')
            block_num += 1

print(f"Wrote target files for subjects {n_collected+1}–{total_sub_num} into {target_dir}/")
print(f"Left subjects 1–{n_collected} unchanged.")


Wrote target files for subjects 6–20 into TargetFiles/
Left subjects 1–5 unchanged.


In [24]:
# read the target files for all subjects
# show the finger trained, speech trained, and untrained sequences for each subject

from collections import Counter

tgt_files = sorted(Path('TargetFiles').glob('SFS2_subj*_group*_block*.tgt'))
all_trials = pd.concat([pd.read_csv(f, sep='\t') for f in tgt_files], ignore_index=True)

rows = []
role_counts = Counter()
for subNum, sub_df in all_trials.groupby('subNum'):
    group = int(sub_df['group'].iloc[0]) + 1
    trained = sub_df[sub_df['isTrain'] == 1]
    finger_trained = sorted(trained.loc[trained['effector'] == 2, 'seq'].astype(str).unique())
    speech_trained = sorted(trained.loc[trained['effector'] == 0, 'seq'].astype(str).unique())

    non_train = sub_df[sub_df['isTrain'] == 0]
    speech_seqs = set(non_train.loc[non_train['effector'] == 0, 'seq'].astype(str))
    finger_seqs = set(non_train.loc[non_train['effector'] == 2, 'seq'].astype(str))
    group_seqs = speech_seqs & finger_seqs
    untrained = sorted(group_seqs - set(finger_trained) - set(speech_trained))

    for s in speech_trained:
        role_counts[(s, 'speech')] += 1
    for s in finger_trained:
        role_counts[(s, 'finger')] += 1
    for s in untrained:
        role_counts[(s, 'untrained')] += 1

    rows.append({
        'subNum': subNum,
        'group': group,
        'finger_trained': finger_trained,
        'speech_trained': speech_trained,
        'untrained': untrained,
    })
    print(f"Subject {subNum} (group {group}):")
    print(f"  Finger trained:  {finger_trained}")
    print(f"  Speech trained:  {speech_trained}")
    print(f"  Untrained:       {untrained}")
    print()

summary = pd.DataFrame(rows)
display(summary)

print("Role balance across subjects (from TargetFiles):")
print(f"{'seq':12}  speech  finger  untrained")
for s in sequences:
    print(f"{s}  {role_counts[(s, 'speech')]:5d}  {role_counts[(s, 'finger')]:6d}  {role_counts[(s, 'untrained')]:8d}")


Subject 1 (group 1):
  Finger trained:  ['34352452141', '54231251343']
  Speech trained:  ['13524232514', '21435235214']
  Untrained:       ['31525423241', '42531451342']

Subject 2 (group 2):
  Finger trained:  ['42531451342', '54231251343']
  Speech trained:  ['13524232514', '21435235214']
  Untrained:       ['31525423241', '34352452141']

Subject 3 (group 3):
  Finger trained:  ['31525423241', '54231251343']
  Speech trained:  ['13524232514', '21435235214']
  Untrained:       ['34352452141', '42531451342']

Subject 4 (group 4):
  Finger trained:  ['21435235214', '54231251343']
  Speech trained:  ['13524232514', '34352452141']
  Untrained:       ['31525423241', '42531451342']

Subject 5 (group 5):
  Finger trained:  ['42531451342', '54231251343']
  Speech trained:  ['13524232514', '34352452141']
  Untrained:       ['21435235214', '31525423241']

Subject 6 (group 6):
  Finger trained:  ['13524232514', '34352452141']
  Speech trained:  ['31525423241', '42531451342']
  Untrained:       

,subNum,group,finger_trained,speech_trained,untrained
0,1,1,"[34352452141, 54231251343]","[13524232514, 21435235214]","[31525423241, 42531451342]"
1,2,2,"[42531451342, 54231251343]","[13524232514, 21435235214]","[31525423241, 34352452141]"
2,3,3,"[31525423241, 54231251343]","[13524232514, 21435235214]","[34352452141, 42531451342]"
3,4,4,"[21435235214, 54231251343]","[13524232514, 34352452141]","[31525423241, 42531451342]"
4,5,5,"[42531451342, 54231251343]","[13524232514, 34352452141]","[21435235214, 31525423241]"
5,6,6,"[13524232514, 34352452141]","[31525423241, 42531451342]","[21435235214, 54231251343]"
6,7,7,"[21435235214, 31525423241]","[42531451342, 54231251343]","[13524232514, 34352452141]"
7,8,8,"[21435235214, 42531451342]","[31525423241, 34352452141]","[13524232514, 54231251343]"
8,9,9,"[13524232514, 31525423241]","[42531451342, 54231251343]","[21435235214, 34352452141]"
9,10,10,"[21435235214, 34352452141]","[31525423241, 54231251343]","[13524232514, 42531451342]"


Role balance across subjects (from TargetFiles):
seq           speech  finger  untrained
13524232514      7       6         7
54231251343      7       7         6
21435235214      7       6         7
34352452141      7       7         6
42531451342      6       7         7
31525423241      6       7         7
